In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# === Load datasets ===
files = [
    "SP1 Season 2020-2021.csv",
    "SP1 Season 2021-2022.csv",
    "SP1 Season 2022-2023.csv",
    "SP1 Season 2023-2024.csv",
    "SP1 Season 2024-2025.csv"
]

dfs = []
for f in files:
    try:
        df = pd.read_csv(f)
        dfs.append(df)
    except Exception as e:
        print(f"⚠️ Skipping {f}: {e}")

data = pd.concat(dfs, ignore_index=True)


data = data.dropna(subset=["HomeTeam", "AwayTeam", "FTR"])

# Label encode teams and results
le_team = LabelEncoder()
data["HomeTeam"] = le_team.fit_transform(data["HomeTeam"])
data["AwayTeam"] = le_team.transform(data["AwayTeam"])

le_ftr = LabelEncoder()
data["FTR"] = le_ftr.fit_transform(data["FTR"])  # 0:H, 1:D, 2:A

# Select numeric features
numeric_features = ["FTHG", "FTAG", "HST", "AST", "HC", "AC", "HF", "AF"]
data = data.dropna(subset=numeric_features)

X = data[["HomeTeam", "AwayTeam"] + numeric_features]
y = data["FTR"]

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

# === Balanced Random Forest ===
model = RandomForestClassifier(
    n_estimators=10,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=2,
    max_features=3,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy: {acc:.3f}")
print("\nClassification Report:\n", classification_report(
    y_test, y_pred, target_names=le_ftr.classes_))

# Save artifacts
artifacts = {
    "model": model,
    "scaler": scaler,
    "le_team": le_team,
    "le_ftr": le_ftr,
    "feature_names": ["HomeTeam", "AwayTeam"] + numeric_features
}

with open("la_liga_model_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("✅ Model artifacts saved successfully → la_liga_model_artifacts.pkl")


✅ Model Accuracy: 0.935

Classification Report:
               precision    recall  f1-score   support

           A       0.93      0.97      0.95       110
           D       0.98      0.81      0.89       106
           H       0.92      0.99      0.95       170

    accuracy                           0.94       386
   macro avg       0.94      0.92      0.93       386
weighted avg       0.94      0.94      0.93       386

✅ Model artifacts saved successfully → la_liga_model_artifacts.pkl
